In [50]:
import numpy as np
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer, util

In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')
model = BertModel.from_pretrained("bert-base-multilingual-uncased")
text = "Replace me by any text you'd like. My name is Luca, I live on the second floor."
encoded_input = tokenizer(text, return_tensors='pt')
output = model(**encoded_input)
output.last_hidden_state.shape

torch.Size([1, 24, 768])

In [13]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

c:\Users\kutzk\anaconda3\envs\seal\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kutzk\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [17]:
embeddings = model.encode(["Hello World. How are you?", "Hallo Welt", "Hola mundo deportivo", "Bye, Moon!"])
model.similarity(embeddings, embeddings)

tensor([[1.0000, 0.7446, 0.4882, 0.3609],
        [0.7446, 1.0000, 0.6723, 0.5307],
        [0.4882, 0.6723, 1.0000, 0.3137],
        [0.3609, 0.5307, 0.3137, 1.0000]])

In [19]:
embeddings = model.encode(["Hello World", "Hallo Welt", "Hola mundo", "Bye, Moon!", "Здравей, свят!"])
similarities = model.similarity(embeddings, embeddings)
similarities

tensor([[1.0000, 0.9429, 0.8880, 0.4558, 0.8133],
        [0.9429, 1.0000, 0.9680, 0.5307, 0.9211],
        [0.8880, 0.9680, 1.0000, 0.4933, 0.8904],
        [0.4558, 0.5307, 0.4933, 1.0000, 0.5486],
        [0.8133, 0.9211, 0.8904, 0.5486, 1.0000]])

In [3]:
datapath = 'data/fr_en/'

def read_entities_map(datapath, filename):
    uri_map = {}
    f = open(datapath+filename, 'r')
    for line in f:
        nr_id, uri = line.split()[0], line.split()[1]
        uri_map[nr_id] = uri
    f.close()
    return uri_map

ent_map1 = read_entities_map(datapath, 'ent_ids_1')
ent_map2 = read_entities_map(datapath, 'ent_ids_2')

In [37]:
from gensim.test.utils import common_texts
common_texts

[['human', 'interface', 'computer'],
 ['survey', 'user', 'computer', 'system', 'response', 'time'],
 ['eps', 'user', 'interface', 'system'],
 ['system', 'human', 'system', 'eps'],
 ['user', 'response', 'time'],
 ['trees'],
 ['graph', 'trees'],
 ['graph', 'minors', 'trees'],
 ['graph', 'minors', 'survey']]

In [135]:
texts = [["John", "likes", "to", "watch", "movies"], ["Mary", "too", "also", "football", "games", "hates"]]

model = Word2Vec(sentences=texts, vector_size=10, window=5, min_count=1, workers=1)
word_vectors = model.wv
word_vectors.save_word2vec_format("data/word2vec.wordvectors")

In [139]:
texts2 = [["Koki", "also", "loves", "football"], ["Koki", "too", "also", "football", "Koki", "hates"], \
          ["Hello", "Koki", "also", "said", "Ani", "Potts", "to", "Koki"]]
model2 = Word2Vec(vector_size=10, window=5, min_count=1, workers=1)
model2.build_vocab(texts2)
model2.wv.vectors_lockf = np.ones(len(model2.wv), dtype='int')
model2.wv.intersect_word2vec_format("data/word2vec.wordvectors")
model2.wv.vectors_lockf = np.ones(len(model2.wv), dtype=np.float32)
model2.train(texts2, total_examples=3, epochs=100)
model2.wv.save_word2vec_format("data/word2vec2.wordvectors")

In [140]:
model2.wv.vectors_lockf

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [141]:
word_vectors2 = model2.wv
word_vectors2['Koki'], word_vectors['John']

(array([-0.00679587,  0.00340094,  0.05154577,  0.09001912, -0.09361933,
        -0.07217073,  0.06560313,  0.09094885, -0.05066276, -0.03757155],
       dtype=float32),
 array([-0.08619688,  0.03665738,  0.05189884,  0.05741938,  0.07466918,
        -0.06167675,  0.01105614,  0.06047282, -0.0284005 , -0.06173522],
       dtype=float32))

In [28]:
len(ent_map1), len(ent_map2)

NameError: name 'ent_map1' is not defined

In [5]:
f = open(datapath + 'fr_att_triples', 'r')
cnt = 0
for line in f:
    print(line)
    cnt += 1
    if cnt > 5:
        break
f.close()

<http://fr.dbpedia.org/resource/Bakir_Izetbegović> <http://fr.dbpedia.org/property/fonction> "président du collège présidentiel de Bosnie-Herzégovine"@fr .

<http://fr.dbpedia.org/resource/FC_Dinamo_Tbilissi> <http://fr.dbpedia.org/property/texte> "championnat soviétique en 1964"@fr .

<http://fr.dbpedia.org/resource/Gaélique_écossais> <http://fr.dbpedia.org/property/wals> "gae"@fr .

<http://fr.dbpedia.org/resource/Ram_(album)> <http://fr.dbpedia.org/property/isbn> "978"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://fr.dbpedia.org/resource/Parti_national_libéral_(Roumanie)> <http://fr.dbpedia.org/property/siège> "Bd. Aviatorilor nr. 86"@fr .

<http://fr.dbpedia.org/resource/Abitibi-Ouest_(circonscription_provinciale)> <http://fr.dbpedia.org/property/date> "2008-10-21"^^<http://www.w3.org/2001/XMLSchema#date> .

